# Electronic Waste Management - Comprehensive EDA

This notebook performs a full exploratory data analysis (EDA) on `electronics_pricing_dataset.csv` to understand:
- data quality and structure
- behavior of pricing and durability features
- category-level patterns (product type, brand, usage pattern)
- a practical **e-waste readiness heuristic** for downstream modeling

In [ ]:
# Core imports
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

plt.style.use('ggplot')
RANDOM_STATE = 42

In [ ]:
# Load dataset
df = pd.read_csv('electronics_pricing_dataset.csv')

print(f'Shape: {df.shape}')
print(f'Rows: {df.shape[0]:,} | Columns: {df.shape[1]}')
print(df.columns)
df.head()

In [ ]:
# Schema and quality checks
print('--- Column types ---')
display(df.dtypes.to_frame('dtype'))

missing = df.isna().sum().sort_values(ascending=False)
duplicates = df.duplicated().sum()

print('--- Data quality summary ---')
print(f'Total missing values: {int(missing.sum())}')
print(f'Duplicate rows: {int(duplicates)}')

if missing.sum() > 0:
    display(missing[missing > 0].to_frame('missing_count'))
else:
    print('No missing values found.')

In [ ]:
# Separate numeric/categorical columns
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include=['object', 'string', 'category']).columns.tolist()

print('Numeric columns:', num_cols)
print('Categorical columns:', cat_cols)

display(df[num_cols].describe().T)

# Useful quantiles for heavy-tailed price behavior
q = df[['Original_Price', 'Current_Price']].quantile([0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])
display(q)

In [ ]:
# Categorical profile
for c in cat_cols:
    print(f'\n=== {c} ===')
    print(f'Unique values: {df[c].nunique()}')
    display(df[c].value_counts(dropna=False).head(15).to_frame('count'))

In [ ]:
# Feature engineering for e-waste analysis
eda = df.copy()

eda['Residual_Value_Pct'] = (eda['Current_Price'] / eda['Original_Price']) * 100
eda['Depreciation_Value'] = eda['Original_Price'] - eda['Current_Price']
eda['Usage_to_Expiry_Ratio'] = eda['Used_Duration'] / eda['Expiry_Years']
eda['Years_to_Expiry'] = eda['Expiry_Years'] - eda['Used_Duration']

# Practical target heuristic (can be replaced by policy-specific rules later)4
eda['Ewaste_Ready'] = ((eda['Used_Duration'] >= eda['Expiry_Years']) |
                       (eda['Condition'] <= 2) |
                       (eda['Current_Price'] <= 1000)).astype(int)

display(eda[['Residual_Value_Pct', 'Depreciation_Value', 'Usage_to_Expiry_Ratio', 'Years_to_Expiry', 'Ewaste_Ready']].describe().T)

In [ ]:
# Univariate numeric distributions
plot_cols = [
    'Original_Price', 'Current_Price', 'Used_Duration', 'Expiry_Years',
    'Condition', 'Build_Quality', 'Residual_Value_Pct', 'Usage_to_Expiry_Ratio'
]

fig, axes = plt.subplots(4, 2, figsize=(14, 16))
axes = axes.flatten()

for i, c in enumerate(plot_cols):
    axes[i].hist(eda[c], bins=35, color='steelblue', alpha=0.85, edgecolor='black')
    axes[i].set_title(f'{c} Distribution')
    axes[i].set_xlabel(c)
    axes[i].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix (numeric only)
corr_cols = [c for c in eda.columns if pd.api.types.is_numeric_dtype(eda[c])]
corr = eda[corr_cols].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(12, 8))
im = ax.imshow(corr.values, cmap='coolwarm', aspect='auto', vmin=-1, vmax=1)

ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=90)
ax.set_yticks(range(len(corr.columns)))
ax.set_yticklabels(corr.columns)
ax.set_title('Correlation Heatmap (Numeric Features)')

cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Correlation')
plt.tight_layout()
plt.show()

display(corr['Current_Price'].sort_values(ascending=False).to_frame('corr_with_Current_Price'))

In [ ]:
# Product-level pricing behavior
prod_stats = (
    eda.groupby('Product_Type', as_index=False)
       .agg(
            count=('Product_Type', 'size'),
            avg_original_price=('Original_Price', 'mean'),
            avg_current_price=('Current_Price', 'mean'),
            median_current_price=('Current_Price', 'median'),
            avg_residual_pct=('Residual_Value_Pct', 'mean'),
            ewaste_ready_rate=('Ewaste_Ready', 'mean')
        )
       .sort_values('median_current_price', ascending=False)
)

display(prod_stats)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

tmp = prod_stats.sort_values('median_current_price', ascending=True)
axes[0].barh(tmp['Product_Type'], tmp['median_current_price'], color='teal')
axes[0].set_title('Median Current Price by Product Type')
axes[0].set_xlabel('Median Current Price')

tmp2 = prod_stats.sort_values('ewaste_ready_rate', ascending=True)
axes[1].barh(tmp2['Product_Type'], tmp2['ewaste_ready_rate'] * 100, color='tomato')
axes[1].set_title('E-waste Ready Rate by Product Type')
axes[1].set_xlabel('E-waste Ready %')

plt.tight_layout()
plt.show()

In [ ]:
# Usage pattern and condition analysis
usage_stats = (
    eda.groupby('Usage_Pattern', as_index=False)
       .agg(
            count=('Usage_Pattern', 'size'),
            avg_used_duration=('Used_Duration', 'mean'),
            avg_current_price=('Current_Price', 'mean'),
            avg_residual_pct=('Residual_Value_Pct', 'mean'),
            ewaste_ready_rate=('Ewaste_Ready', 'mean')
        )
       .sort_values('avg_current_price', ascending=False)
)
display(usage_stats)

cond_stats = (
    eda.groupby('Condition', as_index=False)
       .agg(
            count=('Condition', 'size'),
            avg_current_price=('Current_Price', 'mean'),
            median_current_price=('Current_Price', 'median'),
            avg_residual_pct=('Residual_Value_Pct', 'mean'),
            ewaste_ready_rate=('Ewaste_Ready', 'mean')
        )
       .sort_values('Condition')
)
display(cond_stats)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(cond_stats['Condition'], cond_stats['median_current_price'], marker='o', linewidth=2, color='purple')
ax.set_title('Condition vs Median Current Price')
ax.set_xlabel('Condition')
ax.set_ylabel('Median Current Price')
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Heuristic e-waste readiness deep-dive
overall_rate = eda['Ewaste_Ready'].mean() * 100
rule_a = (eda['Used_Duration'] >= eda['Expiry_Years']).mean() * 100
rule_b = (eda['Condition'] <= 2).mean() * 100
rule_c = (eda['Current_Price'] <= 1000).mean() * 100

print(f'Overall E-waste Ready Rate: {overall_rate:.2f}%')
print(f'Rule A (Used_Duration >= Expiry_Years): {rule_a:.2f}%')
print(f'Rule B (Condition <= 2): {rule_b:.2f}%')
print(f'Rule C (Current_Price <= 1000): {rule_c:.2f}%')

fig, ax = plt.subplots(figsize=(6, 4))
counts = eda['Ewaste_Ready'].value_counts().sort_index()
labels = ['Not Ready (0)', 'Ready (1)']
ax.bar(labels, counts.values, color=['#4c72b0', '#dd8452'])
ax.set_title('E-waste Readiness Class Balance')
ax.set_ylabel('Count')
for i, v in enumerate(counts.values):
    ax.text(i, v + max(counts.values) * 0.01, f'{v:,}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

## EDA Summary (for modeling)

1. The dataset is clean (no missing values, no duplicate rows).
2. `Current_Price` is right-skewed and strongly tied to `Original_Price` and depreciation-related features.
3. `Used_Duration` and `Usage_to_Expiry_Ratio` show clear negative relationship with `Current_Price`.
4. Product categories differ significantly in residual value and e-waste readiness rates.
5. The heuristic `Ewaste_Ready` target is reasonably informative for building a first classification model.

Next step: use these engineered features to train
- a regression model for `Current_Price`
- a classification model for `Ewaste_Ready`